## Plan de feature engineering

À partir de l’EDA réalisée (statistiques descriptives, ADF, ACF/PACF, corrélation, nuages de points, STL), on peut définir un plan de feature engineering structuré autour de trois axes : la dynamique temporelle, les relations physiques entre variables et les régimes de fonctionnement.

### 1. Features temporelles locales

Objectif : capturer la mémoire de court terme mise en évidence par l’ACF/PACF et les tests de stationnarité.

Pour chaque machine (et éventuellement par rôle), sur une série régulièrement rééchantillonnée (par exemple à la minute) :

- **Lags explicites**  
  - `cpu_temp_c_lag_1`, `cpu_temp_c_lag_2`, … (quelques retards courts)  
  - `load_percent_lag_1`, `total_power_w_lag_1`  
  Ces variables permettent aux modèles ou aux règles de tenir compte de l’état récent de la machine sans recalculer à chaque fois l’historique.

- **Fenêtres glissantes**  
  - Moyennes glissantes : `cpu_temp_c_mean_w` sur 3–5 pas de temps, idem pour `load_percent` et `total_power_w`  
  - Écarts-types glissants : `cpu_temp_c_std_w`, `load_percent_std_w`  
  - Deltas : `delta_cpu_temp_c = cpu_temp_c – cpu_temp_c_lag_1`, `delta_load_percent`, `delta_total_power_w`  
  Ces indicateurs capturent la stabilité ou l’agitation locale et la vitesse de variation, ce qui est crucial pour détecter des dérives rapides ou des pics soudains.

- **Résidus par rapport à une tendance courte**  
  En s’appuyant sur la décomposition STL ou sur une moyenne glissante, on peut définir :  
  - `cpu_temp_c_trend` (composante tendance)  
  - `cpu_temp_c_resid = cpu_temp_c – cpu_temp_c_trend`  
  Ce résidu est un bon candidat pour des scores d’anomalie ou des seuils adaptatifs.

### 2. Features de relation physique

Objectif : exploiter les corrélations observées entre charge, température et puissance.

- **Ratios d’efficacité**  
  - `power_per_load = total_power_w / max(load_percent, ε)`  
  - éventuellement `temp_per_load = cpu_temp_c / max(load_percent, ε)`  
  Ces ratios permettent d’identifier des machines “inefficaces” (beaucoup de puissance ou de chaleur pour une charge donnée) par rapport à une baseline machine ou rôle.

- **Écarts à l’environnement**  
  - `temp_delta_ambient = cpu_temp_c – ambient_dc_temp_c`  
  - `temp_delta_external = cpu_temp_c – external_temp_c`  
  Même si, sur cette journée, l’environnement varie peu, ces variables sont structurantes pour la suite : elles isolent la contribution interne (machine) de la composante externe (site).

- **Écarts à la baseline machine/rôle**  
  Pour chaque machine et/ou rôle :  
  - Moyenne historique de référence : `cpu_temp_c_baseline_machine`, `power_baseline_machine`  
  - Écart instantané : `cpu_temp_c – cpu_temp_c_baseline_machine`, `total_power_w – power_baseline_machine`  
  Ces features serviront à définir des règles du type “+X °C au-dessus du comportement normal de cette machine”.

### 3. Features de régime et de structure système

Objectif : refléter le fonctionnement par états (12 / 20 / 46 %) et la topologie du cluster.

- **Catégorisation de la charge**  
  - `load_regime` : variable catégorielle dérivée de `load_percent` (par exemple : `low`, `nominal`, `high`), basée sur la distribution observée (12, 20, 46) et/ou des bornes métier.

- **Compteurs de temps passé en régime**  
  - Pour chaque machine, nombre et durée des séquences consécutives dans chaque régime (`time_in_nominal`, `time_in_high_load`, etc.)  
  Ces compteurs sont utiles pour quantifier la fatigue thermique ou énergétique liée à des périodes prolongées de forte charge.

- **Features de rôle et de cluster**  
  - Encodage du rôle (`role_master`, `role_worker`)  
  - Agrégats au niveau cluster/site : moyenne de `cpu_temp_c` et `total_power_w` sur toutes les machines d’un même cluster au pas de temps courant (contexte global).  
  Cela permet par exemple de comparer une machine à ses pairs au même instant.

### 4. Synthèse pour la suite

Ce plan de feature engineering reste volontairement simple et explicable : il s’appuie sur les propriétés mises en évidence par l’EDA (stationnarité locale, dépendance temporelle, corrélations physiques, fonctionnement par régimes) sans introduire de transformations complexes difficiles à justifier métier. Dans un second temps, une extension plus avancée (Fourier, modèles plus sophistiqués) pourra être envisagée sur des horizons temporels plus longs si des patterns saisonniers riches apparaissent.

Contexte: scénario nominal (pas de stress test, journée de fonctionnement standard).
Objectif: construire les features à partir du dataset brut/normalisé vu dans

Fourier vs STL vs “PID” : pourquoi ces choix ?
Fourier / FFT
La transformée de Fourier ou FFT est très utile quand on veut analyser ou extraire des composantes périodiques fortes, par exemple sur des données d’exploitation collectées sur des jours/semaines avec des cycles jour/nuit, hebdo, etc.

Ici, les données couvrent une seule journée, avec une fréquence horaire, et les variables d’environnement sont quasi constantes. Dans ce contexte:
- on n'a pas encore un cycle jour/nuit complet ou répété à analyser,

- l’intérêt principal est la tendance locale et les régimes de charge, pas la saisonnalité complexe.

STL (décomposition saisonnière par Loess), utilisée sur cpu_temp_c, est précisément conçu pour extraire la tendance, la saisonnalité et le résidu de façon robuste, sans passer par le domaine fréquentiel.

C’est plus lisible et plus facile à expliquer qu’un filtrage spectral “à la FFT” pour ce niveau de granularité.

En bref, Fourier n’est pas nécessairement faux, mais c’est plus lourd à interpréter pour un décideur, STL et des moyennes glissantes répondent déjà bien au besoin de lissage sur cet horizon, la saisonnalité exploitable apparaîtra surtout quand on élargiras la fenêtre temporelle.

PID / filtrage type contrôle
Un PID est un contrôleur (Proportionnel–Intégral–Dérivé) conçu pour agir sur un système (par exemple, ajuster la vitesse du ventilateur en fonction de la température).

Ici, logique d’analyse et d’engineering de features, pas de contrôle en temps réel du système. Utiliser un PID ou un low-pass filter issu du contrôle-commande pour lisser les données d’analyse serait surdimensionné et surtout mal adapté à l’objectif:

- le rôle du PID est de commander une action pour suivre une consigne, pas de préparer des features ;

- la partie dérivée d’un PID nécessite souvent un filtrage spécifique du bruit (Savitzky-Golay, low-pass dédié), ce qui complexifie l’explication sans bénéfice clair pour une première EDA.

Les approches de lissage utilisées en time series/stats sont plutôt les moyennes glissantes, lissage exponentiel, STL/Loess local, etc. Elles sont suffisantes pour réduire le bruit et faire ressortir la tendance, beaucoup plus faciles à justifier dans un contexte de data engineering ou de data science, bien couvertes dans la littérature stats et time series.

En résumé, on fait le bon choix en restant sur STL, moyennes glissantes et résidus pour le lissage, et en ne pas introduire de PID dans le notebook analytique.

In [11]:
import psycopg
import pandas as pd

TS_CONN_STR = "dbname=tsdb user=tsuser password=tspassword host=localhost port=5432"

def load_raw_telemetry(ts_conn_str: str) -> pd.DataFrame:
    query = """
    SELECT
        ts,
        cluster,
        machine,
        role,
        city,
        load_percent,
        cpu_temp_c,
        fan_speed_percent,
        total_power_w,
        ambient_dc_temp_c,
        external_temp_c
    FROM telemetry_normalized
    ORDER BY ts, machine
    """
    with psycopg.connect(ts_conn_str) as conn:
        df = pd.read_sql_query(query, conn, parse_dates=["ts"])
    return df.sort_values(["machine", "ts"]).reset_index(drop=True)

df = load_raw_telemetry(TS_CONN_STR)

C:\Users\emmak\AppData\Local\Temp\ipykernel_4740\3318041465.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, parse_dates=["ts"])


In [12]:
df = df.rename(columns={
    "load_percent": "loadpercent",
    "cpu_temp_c": "cputempc",
    "fan_speed_percent": "fanspeedpercent",
    "total_power_w": "totalpowerw",
    "ambient_dc_temp_c": "ambientdctempc",
    "external_temp_c": "externaltempc",
})

In [19]:
# Dataset de base

def prepare_base_timeseries(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["machine", "ts"]).copy()
    all_machines = []

    for m, g in df.groupby("machine"):
        g = g.set_index("ts").sort_index()

        # on sépare numériques et non numériques
        num_cols = g.select_dtypes(include="number").columns
        cat_cols = [c for c in g.columns if c not in num_cols]

        # resample uniquement sur les numériques
        g_num = g[num_cols].resample("1min").mean().interpolate()

        # pour les colonnes catégorielles, on prend la dernière valeur connue
        g_cat = g[cat_cols].resample("1min").ffill()

        # on recolle
        g_resampled = pd.concat([g_num, g_cat], axis=1)

        # on remet machine en colonne pour la suite
        g_resampled["machine"] = m
        all_machines.append(g_resampled.reset_index())

    return pd.concat(all_machines, ignore_index=True)

In [20]:
df_base = prepare_base_timeseries(df)

In [23]:
df_base.columns.tolist()

['ts',
 'loadpercent',
 'cputempc',
 'fanspeedpercent',
 'totalpowerw',
 'ambientdctempc',
 'externaltempc',
 'cluster',
 'machine',
 'role',
 'city']

In [24]:
print(df_base.columns)

Index(['ts', 'loadpercent', 'cputempc', 'fanspeedpercent', 'totalpowerw',
       'ambientdctempc', 'externaltempc', 'cluster', 'machine', 'role',
       'city'],
      dtype='str')


In [25]:
# Reconstruction des features

def add_lag_features(df: pd.DataFrame, cols, lags):
    df = df.sort_values(["machine", "ts"]).copy()
    for col in cols:
        for lag in lags:
            df[f"{col}_lag{lag}"] = (
                df.groupby("machine")[col].shift(lag)
            )
    return df

def add_rolling_features(df: pd.DataFrame, cols, windows):
    df = df.sort_values(["machine", "ts"]).copy()
    for col in cols:
        for w in windows:
            df[f"{col}_rollmean_{w}"] = (
                df.groupby("machine")[col]
                  .rolling(window=w, min_periods=1)
                  .mean()
                  .reset_index(level=0, drop=True)
            )
    return df

df_fe = (
    df_base
    .pipe(
        add_lag_features,
        cols=["loadpercent", "cputempc", "totalpowerw"],
        lags=[1, 3, 6],
    )
    .pipe(
        add_rolling_features,
        cols=["loadpercent", "cputempc"],
        windows=[5, 15],
    )
    .sort_values(["machine", "ts"])
    .reset_index(drop=True)
)

In [26]:
df_fe.filter(regex="loadpercent|cputempc|totalpowerw").head()

,loadpercent,cputempc,totalpowerw,loadpercent_lag1,loadpercent_lag3,loadpercent_lag6,cputempc_lag1,cputempc_lag3,cputempc_lag6,totalpowerw_lag1,totalpowerw_lag3,totalpowerw_lag6,loadpercent_rollmean_5,loadpercent_rollmean_15,cputempc_rollmean_5,cputempc_rollmean_15
0,12.0,24.260000,171.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.0,12.0,24.260000,24.260000
1,12.0,24.259833,171.6,12.0,NaN,NaN,24.260000,NaN,NaN,171.6,NaN,NaN,12.0,12.0,24.259917,24.259917
2,12.0,24.259667,171.6,12.0,NaN,NaN,24.259833,NaN,NaN,171.6,NaN,NaN,12.0,12.0,24.259833,24.259833
3,12.0,24.259500,171.6,12.0,12.0,NaN,24.259667,24.260000,NaN,171.6,171.6,NaN,12.0,12.0,24.259750,24.259750
4,12.0,24.259333,171.6,12.0,12.0,NaN,24.259500,24.259833,NaN,171.6,171.6,NaN,12.0,12.0,24.259667,24.259667


Deux sources principales des NaN :

- Lags: Pour chaque machine, aux lignes les plus récentes dans le temps (début de série), les features *_lag1, *_lag3, *_lag6 n’existent pas encore.

Exemple: à la première ligne d’une machine, lag1, lag3, lag6 sont tous NaN; à la troisième ligne, lag3 et lag6 sont encore NaN, etc.

- Rolling windows

Même principe: pour une fenêtre de taille 5, les 4 premières lignes ont des moyennes glissantes basées sur moins de points, ou NaN si on met un min_periods plus grand. On a mis min_periods=1, donc ça limite les NaN, mais on en aura quand même sur les lags.

En résumé: ces NaN sont des “bords” temporels normaux. On ne doit pas les remplir arbitrairement si on veut rester fidèle à la causalité; en général on les enlève au moment de préparer le set d’entraînement.

In [27]:
df_fe.to_parquet("telemetry_features_nominal.parquet", index=False)

In [28]:
feature_cols = [c for c in df_fe.columns if "lag" in c or "rollmean" in c]
df_model = df_fe.dropna(subset=feature_cols).reset_index(drop=True)
df_model.to_parquet("telemetry_features_nominal_clean.parquet", index=False)